In [11]:
pip install -U transformers

In [12]:
!pip install -q gradio transformers torch reportlab

In [13]:
import gradio as gr
import re
from datetime import datetime
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

In [14]:
schema = {
    "users": ["id","name","email","signup_date","age","country","status"],
    "orders": ["id","user_id","product_name","amount","order_date","status"],
    "products": ["id","name","price","category","stock"],
    "transactions": ["id","user_id","amount","date","type"]
}

In [15]:
dangerous_keywords = ["DROP","DELETE","ALTER","TRUNCATE","INSERT","UPDATE","EXEC"]

def is_safe(text):
    text_upper = text.upper()

    for word in dangerous_keywords:
        if word in text_upper:
            return False

    if ";" in text or "--" in text:
        return False

    return True

In [16]:
def get_table_name(text):
    text = text.lower()

    if "user" in text or "customer" in text:
        return "users"

    if "order" in text or "purchase" in text:
        return "orders"

    if "product" in text or "item" in text:
        return "products"

    if "transaction" in text or "payment" in text:
        return "transactions"

    return "users"

In [17]:
def extract_conditions(text):

    text = text.lower()

    if "last month" in text:
        return "DATE(signup_date) >= DATE('now','-1 month')"

    if "last week" in text:
        return "DATE(date) >= DATE('now','-7 days')"

    amount_match = re.search(r'greater than (\d+)', text)

    if amount_match:
        value = amount_match.group(1)
        return f"amount > {value}"

    return ""

In [18]:
def generate_sql(user_input):

    if not is_safe(user_input):
        return "❌ Unsafe query detected"

    table = get_table_name(user_input)

    columns = "*"

    condition = extract_conditions(user_input)

    query = f"SELECT {columns} FROM {table}"

    if condition:
        query += f" WHERE {condition}"

    query += " LIMIT 10"

    return query

In [19]:
history = []

def export_pdf():

    filename = f"sql_report_{datetime.now().strftime('%Y%m%d%H%M%S')}.pdf"

    c = canvas.Canvas(filename, pagesize=letter)

    y = 750

    for item in history:
        c.drawString(50,y,item)
        y -= 20

    c.save()

    return filename

In [20]:
def chat(user_input, chat_history):

    sql = generate_sql(user_input)

    history.append(sql)

    chat_history.append((user_input, sql))

    return "", chat_history

with gr.Blocks() as demo:

    gr.Markdown("# SQL Query Generator")

    chatbot = gr.Chatbot()

    msg = gr.Textbox(label="Ask in English")

    clear = gr.Button("Clear")

    msg.submit(chat,[msg,chatbot],[msg,chatbot])

    clear.click(lambda:None,None,chatbot,queue=False)

demo.launch(share=True)

/tmp/ipykernel_216/1501594242.py:15: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()
/tmp/ipykernel_216/1501594242.py:15: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot()


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5110457cc689eab1ea.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
